# ARC AGI 3 Colab Training

This notebook does four things:

1. Copies the project from Google Drive to the Colab local disk
2. Installs the required dependencies
3. Runs public environment collection and model training
4. Copies code from `ARC Prize 2026 - ARC-AGI-3/` and writes logs, validation outputs, and checkpoints to `ARC Prize 2026_AGI_3/Training_Output/<timestamp>/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3')
DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/ARC Prize 2026_AGI_3/Training_Output')
DRIVE_COLLECTION_BASE = Path('/content/drive/MyDrive/ARC Prize 2026_AGI_3/Collection_Cache')
LOCAL_WORKDIR = Path('/content/ARC Prize 2026 - ARC-AGI-3')
RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_COLLECTION_BASE.mkdir(parents=True, exist_ok=True)

print('Drive project root:', DRIVE_PROJECT_ROOT)
print('Drive output base:', DRIVE_OUTPUT_BASE)
print('Drive collection base:', DRIVE_COLLECTION_BASE)
print('Local workdir:', LOCAL_WORKDIR)
print('Output root:', OUTPUT_ROOT)


In [ ]:
!rm -rf "$LOCAL_WORKDIR"
!mkdir -p "$LOCAL_WORKDIR"
!rsync -a --delete --exclude '.git' "$DRIVE_PROJECT_ROOT/" "$LOCAL_WORKDIR/"
%cd "$LOCAL_WORKDIR"


In [ ]:
import os, sys, subprocess

def run(cmd):
    print('>>>', cmd)
    subprocess.check_call(cmd, shell=True)

run('python -m pip install -U pip wheel setuptools')
run('python -m pip install -U torch torchvision torchaudio')

try:
    run('python -m pip install -U arc-agi==0.9.8 arcengine==0.9.3')
except Exception:
    print('PyPI install failed, trying local wheels...')
    run('python -m pip install arc_agi_3_wheels/*.whl')

run('python - <<\'PY\'\nimport torch\nprint("torch", torch.__version__)\nprint("cuda", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("device", torch.cuda.get_device_name(0))\nPY')


In [ ]:
HARDWARE_PROFILE = 'a100'  # Training profile. Switch to 'h100' only if needed.
COLLECT_PROFILE = 'a100'  # For smoke tests use 'cpu_debug'.
COLLECT_TAG = 'public_search_a100_v1'
RUN_COLLECTION = False  # Set to True only when you want to build or refresh the cached public trajectories.
COLLECT_STEPS = 96
COLLECT_WORKERS = 8  # Use more only if the runtime has the CPU cores and RAM to support it.
COLLECT_GAMES = None  # Example: 'ls20,ar25'
COLLECT_EPISODES_PER_GAME = None  # Example: 8 for a cheaper first pass
COLLECT_BEAM_WIDTH = None  # Example: 4
COLLECT_BRANCH_FACTOR = None  # Example: 6
CHECKPOINT_EVERY_STEPS = 100
RESUME_CHECKPOINT = None  # Example: OUTPUT_ROOT / 'checkpoints' / 'last.pth'

COLLECT_ROOT = DRIVE_COLLECTION_BASE / COLLECT_TAG
TRAIN_DATA_PATH = COLLECT_ROOT / 'collected' / 'episodes.jsonl.gz'
collect_optional_args = []
collect_optional_args.append(f'--workers {COLLECT_WORKERS}')
if COLLECT_GAMES:
    collect_optional_args.append(f'--games {COLLECT_GAMES}')
if COLLECT_EPISODES_PER_GAME is not None:
    collect_optional_args.append(f'--episodes-per-game {COLLECT_EPISODES_PER_GAME}')
if COLLECT_BEAM_WIDTH is not None:
    collect_optional_args.append(f'--beam-width {COLLECT_BEAM_WIDTH}')
if COLLECT_BRANCH_FACTOR is not None:
    collect_optional_args.append(f'--branch-factor {COLLECT_BRANCH_FACTOR}')
collect_optional = '' if not collect_optional_args else ' \\\n  ' + ' \\\n  '.join(collect_optional_args)

if RUN_COLLECTION:
    collect_cmd = f'''python -m src.collect \
  --project-root "{LOCAL_WORKDIR}" \
  --output-root "{COLLECT_ROOT}" \
  --hardware-profile {COLLECT_PROFILE} \
  --seeds 0,1,2,3 \
  --max-steps {COLLECT_STEPS}{collect_optional}'''
    run(collect_cmd)
else:
    print('Skipping collection. Reusing cached trajectories from:', TRAIN_DATA_PATH)

if not TRAIN_DATA_PATH.exists():
    raise FileNotFoundError(f'Collected data not found: {TRAIN_DATA_PATH}. Set RUN_COLLECTION = True first.')

print('Training data path:', TRAIN_DATA_PATH)


In [ ]:
resume_arg = '' if RESUME_CHECKPOINT is None else f' \\\n  --resume "{RESUME_CHECKPOINT}"'

train_cmd = f'''python -m src.train \
  --project-root "{LOCAL_WORKDIR}" \
  --data "{TRAIN_DATA_PATH}" \
  --output-dir "{OUTPUT_ROOT}" \
  --hardware-profile {HARDWARE_PROFILE} \
  --max-steps 192 \
  --online-val-games 5 \
  --checkpoint-every-steps {CHECKPOINT_EVERY_STEPS}{resume_arg}'''

run(train_cmd)


In [ ]:
eval_cmd = f'''python -m src.evaluate \
  --project-root "{LOCAL_WORKDIR}" \
  --checkpoint "{OUTPUT_ROOT / 'checkpoints' / 'best.pth'}" \
  --output "{OUTPUT_ROOT / 'public_eval.json'}" \
  --split val'''

run(eval_cmd)


In [ ]:
import pandas as pd
from pathlib import Path

metrics_path = OUTPUT_ROOT / 'metrics.csv'
display(pd.read_csv(metrics_path).tail())
print('Best checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'best.pth')
print('Public eval:', OUTPUT_ROOT / 'public_eval.json')
